# Predicting Student Academic Outcomes

**Summative Project — Introduction to Machine Learning**

Traditional ML (scikit-learn) vs Deep Learning (TensorFlow/Keras) on the **Open University Learning Analytics Dataset (OULAD)**.

**Target:** `final_result` — Distinction, Pass, Fail, Withdrawn

## 1. Setup

In [ ]:
GITHUB_REPO = "https://github.com/Beulla/model_training_and_evaluation"
OULAD_RELEASE_URL = None
INCLUDE_VLE = True

In [ ]:
from pathlib import Path

if not Path("src").exists():
    !git clone {GITHUB_REPO}.git
    %cd model_training_and_evaluation

!pip install -q -r requirements.txt

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from src.data_loader import RAW_DIR, build_modeling_dataset, ensure_data
from src.preprocessing import build_preprocessor, prepare_features, split_data
from src.evaluation import (
    compute_multiclass_auc,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_learning_curve_keras,
    plot_roc_ovr,
    print_classification_summary,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

## 2. Data Loading

In [ ]:
ensure_data(RAW_DIR, release_url=OULAD_RELEASE_URL)

df = build_modeling_dataset(RAW_DIR, include_vle=INCLUDE_VLE)
print(f"Dataset shape: {df.shape}")
print(f"Meets 20k requirement: {len(df) >= 20000}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
class_counts = df["final_result"].value_counts()
print(class_counts)
print(f"\nClass proportions:\n{(class_counts / len(df) * 100).round(1)}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="final_result", order=class_counts.index, hue="final_result", legend=False, ax=ax)
ax.set_title("Distribution of Final Results")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ["gender", "highest_education", "age_band"]):
    ct = pd.crosstab(df[col], df["final_result"], normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")
    ax.set_title(f"Outcome by {col}")
    ax.set_ylabel("Percentage")
    ax.legend(title="Result", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
feature_df, y, label_encoder, cat_cols, num_cols = prepare_features(df)
class_names = list(label_encoder.classes_)
print(f"Classes: {class_names}")
print(f"Categorical: {cat_cols}")
print(f"Numeric: {num_cols}")

splits = split_data(feature_df, y, random_state=RANDOM_STATE)
for key, val in splits.items():
    print(f"{key}: {val.shape if hasattr(val, 'shape') else len(val)}")

In [ ]:
preprocessor = build_preprocessor(cat_cols, num_cols)

X_train = preprocessor.fit_transform(splits["X_train"])
X_val = preprocessor.transform(splits["X_val"])
X_test = preprocessor.transform(splits["X_test"])

feature_names = preprocessor.get_feature_names_out()
print(f"Transformed feature count: {X_train.shape[1]}")

## 5. Traditional Machine Learning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ml_models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"C": [0.1, 1.0, 10.0]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [100, 200], "max_depth": [10, 20, None]},
    ),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {"max_depth": [5, 10], "learning_rate": [0.05, 0.1]},
    ),
    "SVM": (
        SVC(probability=True, random_state=RANDOM_STATE),
        {"C": [0.1, 1.0], "kernel": ["rbf", "linear"]},
    ),
}

ml_results = {}
best_ml_model = None
best_ml_score = -1

for name, (estimator, param_grid) in ml_models.items():
    print(f"\nTuning {name}...")
    search = GridSearchCV(
        estimator,
        param_grid,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
    )
    search.fit(X_train, splits["y_train"])
    y_pred = search.predict(X_test)
    y_proba = search.predict_proba(X_test)

    report = print_classification_summary(
        splits["y_test"], y_pred, class_names, model_name=name
    )
    auc = compute_multiclass_auc(splits["y_test"], y_proba, len(class_names))
    print(f"Macro ROC-AUC: {auc:.4f}")
    print(f"Best params: {search.best_params_}")

    ml_results[name] = {
        "model": search.best_estimator_,
        "f1_macro": report["macro avg"]["f1-score"],
        "auc": auc,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

    if report["macro avg"]["f1-score"] > best_ml_score:
        best_ml_score = report["macro avg"]["f1-score"]
        best_ml_name = name
        best_ml_model = search.best_estimator_

In [ ]:
comparison = pd.DataFrame(
    {name: {"F1 (macro)": r["f1_macro"], "ROC-AUC (macro)": r["auc"]} for name, r in ml_results.items()}
).T.sort_values("F1 (macro)", ascending=False)
print(comparison.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
comparison.plot(kind="bar", ax=ax)
ax.set_title("Traditional ML Model Comparison")
ax.set_ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
best = ml_results[best_ml_name]
fig, ax = plt.subplots(figsize=(8, 6))
plot_confusion_matrix(splits["y_test"], best["y_pred"], class_names,
                      title=f"{best_ml_name} — Confusion Matrix", ax=ax)
plt.show()

plot_roc_ovr(splits["y_test"], best["y_proba"], class_names,
             title=f"{best_ml_name} — ROC Curves")
plt.show()

In [ ]:
if hasattr(best_ml_model, "feature_importances_"):
    plot_feature_importance(
        best_ml_model.feature_importances_,
        list(feature_names),
        title=f"{best_ml_name} — Top Features",
    )
    plt.show()

## 6. Deep Learning

In [ ]:
def make_tf_dataset(X, y, batch_size=256, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.int32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(y), seed=RANDOM_STATE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(X_train, splits["y_train"], shuffle=True)
val_ds = make_tf_dataset(X_val, splits["y_val"])
test_ds = make_tf_dataset(X_test, splits["y_test"])

n_features = X_train.shape[1]
n_classes = len(class_names)

dl_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n_features,)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(n_classes, activation="softmax"),
], name="student_outcome_nn")

dl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
dl_model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss"),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, monitor="val_loss"),
]

history = dl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks,
    verbose=1,
)

plot_learning_curve_keras(history, title="Neural Network Learning Curves")
plt.show()

In [ ]:
dl_loss, dl_acc = dl_model.evaluate(test_ds, verbose=0)
print(f"Test Loss: {dl_loss:.4f} | Test Accuracy: {dl_acc:.4f}")

dl_proba = dl_model.predict(X_test.astype(np.float32), verbose=0)
dl_pred = np.argmax(dl_proba, axis=1)

dl_report = print_classification_summary(
    splits["y_test"], dl_pred, class_names, model_name="Deep Learning (NN)"
)
dl_auc = compute_multiclass_auc(splits["y_test"], dl_proba, n_classes)
print(f"Macro ROC-AUC: {dl_auc:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
plot_confusion_matrix(splits["y_test"], dl_pred, class_names,
                      title="Deep Learning — Confusion Matrix", ax=ax)
plt.show()

plot_roc_ovr(splits["y_test"], dl_proba, class_names, title="Deep Learning — ROC Curves")
plt.show()

## 7. Final Comparison & Model Export

In [ ]:
final_comparison = pd.DataFrame({
    "Model": [best_ml_name, "Deep Learning (NN)"],
    "F1 (macro)": [best_ml_score, dl_report["macro avg"]["f1-score"]],
    "ROC-AUC (macro)": [ml_results[best_ml_name]["auc"], dl_auc],
    "Accuracy": [
        ml_results[best_ml_name]["model"].score(X_test, splits["y_test"]),
        dl_acc,
    ],
})
print(final_comparison.round(4).to_string(index=False))

from google.colab import drive
drive.mount("/content/drive")

MODELS_DIR = Path("/content/drive/MyDrive/ml_summative_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

ml_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_ml_model),
])
joblib.dump(ml_pipeline, MODELS_DIR / "best_traditional_ml.joblib")
dl_model.save(MODELS_DIR / "best_deep_learning.keras")
print(f"Models saved to {MODELS_DIR}")